# Tutorial 2: 1,000 Hz Incipient Slip Detection & Discrete Admittance Control

This tutorial walks through the mathematical pipeline running on the palm MCU at **1,000 Hz**:
1. **Zero-Window-Delay Biquad Bandpass Filter (50–300 Hz)** isolating stick-slip tangential micro-vibrations ($< 1.2\text{ ms}$ group delay).
2. **Howe-Cutkosky Ellipsoidal Friction Margin** $\eta_{\text{fric}}$ combined with EWMA vibration energy $E_{\text{vib}}$ to form a continuous slip risk index $\sigma_{\text{slip}} \in [0, 1]$.
3. **Decoupled Discrete-Time Admittance Law** with normal stiffening ($\zeta_n \ge 1.0$) and **bumpless slow-loop handoff**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from bimodal_tactile.sensors import Hall6DArraySimulator, Hall6DConfig
from bimodal_tactile.control import IncipientSlipDetector, DiscreteAdmittanceController

hall = Hall6DArraySimulator(np.zeros((1, 3)), np.array([1], dtype=np.int32), Hall6DConfig(mu_static=0.6))
detector = IncipientSlipDetector()
controller = DiscreteAdmittanceController()

fn_ref, actual_fn, k_env = 1.5, 1.5, 1200.0
logs = []
for k in range(500):
    ft = 0.4 + (1.1 * min(1.0, max(0.0, (k - 80) / 220.0)))
    if k > 0 and k % 17 == 0:
        fn_ref = 0.85 * fn_ref + 0.15 * controller.last_fn_desired
        controller.bumpless_slow_loop_handoff(fn_ref)
    meas = hall.step(np.array([[ft, 0.0, actual_fn, 0.0, 0.0, 0.002]]), dt=0.001)[0]
    slip = detector.step(meas)
    out = controller.step(np.array([ft, 0.0, fn_ref]), meas[:3], slip)
    actual_fn = max(0.1, fn_ref + k_env * out['delta_x_contact'][2])
    logs.append((k, ft, actual_fn, slip['eta_fric'], slip['sigma_slip']))

arr = np.array(logs)
fig, ax = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
ax[0].plot(arr[:, 0], arr[:, 1], label='Tangential Payload Ft (N)')
ax[0].plot(arr[:, 0], arr[:, 2], label='Closed-Loop Normal Grip Fn (N)', lw=2)
ax[0].legend(); ax[0].grid(True, alpha=0.3)
ax[1].plot(arr[:, 0], arr[:, 3], label='Ellipsoidal Friction Utilization eta_fric')
ax[1].axhline(1.0, color='r', ls='--', label='Macro-Slip Limit')
ax[1].plot(arr[:, 0], arr[:, 4], label='Slip Risk sigma_slip')
ax[1].set_xlabel('Time (ms)'); ax[1].legend(); ax[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()